# BPL Linear Algebra

In [1]:
!pip install -U basedpl

In [2]:
%load_ext basedpl.notebooks

In [3]:
import basedpl
basedpl.__version__

'0.1.91'

## Linear algebra in BPL, part 1: matrices are just arrays

Plan: build matrices, see how the leading axis drives reductions and broadcasting, then use inner product (`+.×`), outer product (`⊗`) and matrix divide (`⌹`). Each step ends with a small exercise for you.

To create a matrix, put each row on a new line, or separate rows with the **diamond** character `◇` on one line. For example, `[1 2 3◇4 5 6]` creates a 2×3 matrix. (`∘` is **jot**, a different symbol.)

- **Shape:** `ρM` gives the size of each axis. For a matrix with 2 rows and 3 columns, it returns `2 3`.
- **Transpose:** `⍉M` swaps rows and columns, turning a 2×3 matrix into a 3×2 matrix.

In [4]:
%%bpl
A←[2 1
   1 3]
M←[1 2 3◇4 5 6]
ρM
M
⍉M

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

[2 3]ₓ

┌→────┐
↓1 2 3│
│4 5 6│
└~────┘

┌→──┐
↓1 4│
│2 5│
│3 6│
└~──┘

### The leading axis decides

`M` has shape `2 3`: axis 0 has length 2 (rows), axis 1 has length 3 (columns). BPL treats the **first** axis as primary:

- `+⌿` reduces along axis 0, so it sums down the columns. `+/` is the same thing applied at rank 1, so it sums each row.
- Broadcasting matches **leading** axes, so `M+10 20` adds 10 to row 0 and 20 to row 1.
- To match the last axis instead, apply at rank 1 with `⍤1`. Note the brackets: a bare literal after `⍤1 ` would join the operand strand.

In [5]:
%%bpl
M
+⌿M
+/M
M+10 20
M +⍤1 [10 20 30]

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓1 2 3│
│4 5 6│
└~────┘

5 7 9

6 15

┌→───────┐
↓11 12 13│
│24 25 26│
└~───────┘

┌→───────┐
↓11 22 33│
│14 25 36│
└~───────┘

### Inner product: `+.×`

Read `+.×` as **“multiply matching numbers, then add the products.”** For example:

```python
1 2 3 +.× 1 0 1  =  1×1 + 2×0 + 3×1  =  4
```

For matrix multiplication, BPL repeats that calculation between each **row of the left matrix** and each **column of the right matrix**.

- `A+.×A` multiplies the rows of `A` by its columns. Its top-left result is `2×2 + 1×1 = 5`.
- `M+.×1 0 1` does the vector calculation for each row of `M`, giving `4 10`. In BPL, the vector `1 0 1` has shape `3` and no row/column orientation. It plays the role of a 3×1 column vector here, but the result has shape `2`, not `2 1`.
- `M+.×⍉M` compares every row of `M` with every row of `M`; the transpose turns the second set of rows into columns.

More generally, `f.g` combines matching values with `g`, then combines those results with `f`. Thus `+.×` uses `×` first and `+` second.

In [6]:
%%bpl
"A:" A
"M:" M
A+.×A
M+.×1 0 1
ρ(M+.×1 0 1)
M+.×⍉M

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→───────────┐
│ "A:" ┌→──┐ │
│      ↓2 1│ │
│      │1 3│ │
│      └~──┘ │
└∊───────────┘

┌→─────────────┐
│ "M:" ┌→────┐ │
│      ↓1 2 3│ │
│      │4 5 6│ │
│      └~────┘ │
└∊─────────────┘

┌→───┐
↓5  5│
│5 10│
└~───┘

4 10

[2]ₓ

┌→────┐
↓14 32│
│32 77│
└~────┘

Before using an outer product to build an identity matrix, start with the basic pattern: **combine every item on the left with every item on the right**.

For vectors of lengths 2 and 3, the result is a 2×3 matrix:

In conventional matrix notation, you can picture the left vector as a 2×1 column and the right vector as a 1×3 row, producing a 2×3 result. Strictly speaking, BPL vectors have shapes `2` and `3`, with no row/column orientation; the outer product creates the two result axes.

- `1 2 +⊗ 10 20 30` adds every left number to every right number. The first row uses `1`; the second uses `2`.
- `2 3 ×⊗ 4 5` does the same with multiplication.

So an outer product does not match corresponding positions like an inner product. It makes **all possible pairs**. The next example uses this same idea first with multiplication, then with equality to construct an identity matrix.

In [7]:
%%bpl
1 2 +⊗ 10 20 30
2 3 ×⊗ 4 5

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→───────┐
↓11 21 31│
│12 22 32│
└~───────┘

┌→────┐
↓ 8 10│
│12 15│
└~────┘

### Outer product `⊗`, and an identity matrix

`ι3` makes `0 1 2`.

- `=⊗⍨ι3`: `⍨` uses `0 1 2` on both sides, and `=⊗` compares every possible pair. Each number equals only itself, so `$t` appears on the diagonal and `$f` everywhere else.
- `I←+=⊗⍨ι3`: monadic `+` converts `$t/$f` to `1/0`, then `I←` stores the resulting identity matrix in `I`.
- The final `I` simply displays the stored matrix.

In [8]:
%%bpl
1 2 3 ×⊗ 4 5
=⊗⍨ι3
I←+=⊗⍨ι3
I

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓ 4  5│
│ 8 10│
│12 15│
└~────┘

┌→───────┐
↓$t $f $f│
│$f $t $f│
│$f $f $t│
└$───────┘

┌→────┐
↓1 0 0│
│0 1 0│
│0 0 1│
└ₓ────┘

### Matrix divide `⌹`: inverses and solving equations

`⌹` is BPL’s **matrix divide** symbol. We will use it in two ways:

- `⌹A` finds the **inverse** of `A`. An inverse undoes matrix multiplication, so `A+.×⌹A` should give the identity matrix.
- `b⌹A` solves the equation `A+.×x=b`. It finds the unknown vector `x`.

There is one numerical detail to watch. Our original `A` contains ordinary floating-point numbers, so its answers are approximate. Adding `ᵣ` to a matrix asks BPL to use exact numbers instead.

#### 1. Find an approximate inverse

Start with `⌹A`. BPL returns the matrix which undoes multiplication by `A`. The decimal entries are approximate because `A` uses floating-point numbers.

In [9]:
%%bpl
⌹A

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────┐
↓ 0.6 ¯0.2│
│¯0.2  0.4│
└~────────┘

#### 2. Check the approximate inverse

Multiply `A` by its inverse. We expect the identity matrix: ones on the diagonal and zeros everywhere else.

The tiny value `¯5.551115123ₑ¯17` is floating-point rounding noise. For practical purposes, it is zero.

In [10]:
%%bpl
A+.×⌹A

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→─────────────────┐
↓ 1               0│
│¯5.551115123ₑ¯17 1│
└~─────────────────┘

#### 3. Make an exact copy of the matrix

Rebuild the same matrix and add `ᵣ` to mark every number as exact. Store this matrix in `Ax`.

Assignment is silent, so this cell displays no result.

In [11]:
%%bpl
Ax←[2 1◇1 3]ᵣ

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

#### 4. Find the exact inverse

Now invert `Ax`. Because `Ax` is exact, BPL returns exact fractions instead of approximate decimals.

`3ᵣ5` means $3/5$. `¯1ᵣ5` means $-1/5$.

In [12]:
%%bpl
⌹Ax

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────┐
↓ 3ᵣ5 ¯1ᵣ5│
│¯1ᵣ5  2ᵣ5│
└+────────┘

#### 5. Check the exact inverse

Multiply the exact matrix by its exact inverse. The result is now an exact identity matrix, with no rounding noise.

In [13]:
%%bpl
Ax+.×⌹Ax

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓1ᵣ 0ᵣ│
│0ᵣ 1ᵣ│
└+────┘

#### 6. Solve a matrix equation

`5 10⌹A` finds the vector `x` that makes `A+.×x` equal `5 10`.

The answer is `1 3`. We can check it by multiplying: `A+.×1 3` gives `5 10`.

In [14]:
%%bpl
5 10⌹A
A+.×1 3

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

1 3

5 10

### Your turn

1. **Predict, then run:** what does `M+10 20 30` do, and why?
2. Name the product: `mm←+.×`. Make a non-symmetric `B` and show `A mm B` and `B mm A` differ.

## 1.

Let's remind ourselves what `M` is.

In [15]:
%%bpl
M

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓1 2 3│
│4 5 6│
└~────┘

I predict adding `10 20 30` to `M` should add `10` to row `0`, `20` to row `1`, and `30` to row `3` (broadcast).

In [16]:
%%bpl
M+10 20 30

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

BplError: LENGTH ERROR: lengths 2 and 3 don't agree
 --> <input>:1:2
M+10 20 30
 ^

In [ ]:
%%prompt
But broadcasting doesnt work here so maybe that was a trick question?

Exactly—it tests the **leading-axis** rule.

- `M` has shape `2 3`.
- `10 20 30` has shape `3`.
- Ordinary `+` tries to match that vector against `M`’s **leading axis**, whose length is `2`.
- Since `2 ≠ 3`, BPL raises a `LENGTH ERROR`.

To add the three values across each row, apply `+` at rank 1:

```apl
M +⍤1 [10 20 30]
```

This should produce:

```text
11 22 33
14 25 36
```

By contrast, `M+10 20` works because its two values match the two rows, adding `10` to row 0 and `20` to row 1.

```json {.usage}
{"model": "gpt-5.6-sol", "prompt_tokens": 42929, "completion_tokens": 316, "total_tokens": 43245, "cached_tokens": 38272, "reasoning_tokens": 138, "cost": 0.006676350000000001}
```


## 2.

`mm←+.×` is a standard dot product?

In [17]:
%%bpl
A

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──┐
↓2 1│
│1 3│
└~──┘

In [18]:
%%bpl
mm←+.×
A
B←[4 5◇6 7]
B

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──┐
↓2 1│
│1 3│
└~──┘

┌→──┐
↓4 5│
│6 7│
└~──┘

In [19]:
%%bpl
A mm B
B mm A

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓14 17│
│22 26│
└~────┘

┌→────┐
↓13 19│
│19 27│
└~────┘

In [ ]:
%%prompt
check ans 1 and 2.

Both answers are correct, with one correction to your original prediction in **1**.

1. **Broadcasting:** `M+10 20 30` gives a `LENGTH ERROR`, because ordinary broadcasting matches the vector against the leading axis of `M`. That axis has length 2, but the vector has length 3. You correctly identified the trick after running it.

   To add by columns instead:

   ```apl
   M +⍤1 [10 20 30]
   ```

2. **Matrix multiplication:** Correct. `mm←+.×` names the plus-times inner product:
   - for two vectors, it is the usual dot product;
   - for two matrices, it is matrix multiplication.

   Your `B` is non-symmetric because its off-diagonal entries differ: \(5\ne6\). Your results are also correct:

   \[
   AB=\begin{bmatrix}14&17\\22&26\end{bmatrix},
   \qquad
   BA=\begin{bmatrix}13&19\\19&27\end{bmatrix}.
   \]

   Therefore \(AB\ne BA\), demonstrating that matrix multiplication is generally **not commutative**.

```json {.usage}
{"model": "gpt-5.6-sol", "prompt_tokens": 43451, "completion_tokens": 427, "total_tokens": 43878, "cached_tokens": 38272, "reasoning_tokens": 166, "cost": 0.006837900000000001}
```


## Linear algebra in BPL, part 2: matrix powers and iteration

Plan: use the power operator `⍣` to repeat a matrix product. We'll generate Fibonacci numbers and find where a Markov chain settles.

### The power operator `⍣` has four forms

- `f⍣n` applies `f` n times.
- `f⍣(ιn)` keeps the history: one result for each count 0 … n-1. Count 0 is the starting value.
- `f⍣≡` keeps applying `f` until the result stops changing (a fixed point).
- `f⍣[≡]` keeps every result until the fixed point.

In [20]:
%%bpl
x←1
2×⍣3 x
2×⍣(ι5) x
{⌊ω÷2}⍣≡ 100
{⌊ω÷2}⍣[≡] 100  ⍝ keep every result until the fixed point

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

8

1 2 4 8 16

0ₓ

100 50ₓ 25ₓ 12ₓ 6ₓ 3ₓ 1ₓ 0ₓ 0ₓ

### Fibonacci from a matrix

#### 1. Powers of F

With `F←[1 1◇1 0]`, the matrix power Fⁿ is `[F(n+1) F(n)◇F(n) F(n-1)]`. `F+.×⍣9 F` multiplies F into itself 9 more times, which gives F¹⁰.

In [21]:
%%bpl
F←[1 1◇1 0]ᵣ
F+.×⍣9 F
F+.×⍣(ι10) F  ⍝ keep F¹ through F¹⁰

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──────┐
↓89ᵣ 55ᵣ│
│55ᵣ 34ᵣ│
└+──────┘

┌→─────────────────────────────────────────────────────────────────────────────────────────┐
│ ┌→────┐ ┌→────┐ ┌→────┐ ┌→────┐ ┌→────┐ ┌→─────┐ ┌→──────┐ ┌→──────┐ ┌→──────┐ ┌→──────┐ │
│ ↓1ᵣ 1ᵣ│ ↓2ᵣ 1ᵣ│ ↓3ᵣ 2ᵣ│ ↓5ᵣ 3ᵣ│ ↓8ᵣ 5ᵣ│ ↓13ᵣ 8ᵣ│ ↓21ᵣ 13ᵣ│ ↓34ᵣ 21ᵣ│ ↓55ᵣ 34ᵣ│ ↓89ᵣ 55ᵣ│ │
│ │1ᵣ 0ᵣ│ │1ᵣ 1ᵣ│ │2ᵣ 1ᵣ│ │3ᵣ 2ᵣ│ │5ᵣ 3ᵣ│ │ 8ᵣ 5ᵣ│ │13ᵣ  8ᵣ│ │21ᵣ 13ᵣ│ │34ᵣ 21ᵣ│ │55ᵣ 34ᵣ│ │
│ └+────┘ └+────┘ └+────┘ └+────┘ └+────┘ └+─────┘ └+──────┘ └+──────┘ └+──────┘ └+──────┘ │
└∊─────────────────────────────────────────────────────────────────────────────────────────┘

#### 2. Keep every step

Multiply the starting vector by `F`, and use `⍣(ι12)` to keep every step. The result is a vector containing 12 two-item vectors.

In [22]:
%%bpl
fibs←(F+.×)⍣(ι12) [1 0]ᵣ
fibs
ρfibs    ⍝ outer shape: 12
ρ¨fibs   ⍝ each nested item has shape 2

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ ┌→────┐ ┌→────┐ ┌→────┐ ┌→────┐ ┌→────┐ ┌→────┐ ┌→─────┐ ┌→──────┐ ┌→──────┐ ┌→──────┐ ┌→──────┐ ┌→───────┐ │
│ │1ᵣ 0ᵣ│ │1ᵣ 1ᵣ│ │2ᵣ 1ᵣ│ │3ᵣ 2ᵣ│ │5ᵣ 3ᵣ│ │8ᵣ 5ᵣ│ │13ᵣ 8ᵣ│ │21ᵣ 13ᵣ│ │34ᵣ 21ᵣ│ │55ᵣ 34ᵣ│ │89ᵣ 55ᵣ│ │144ᵣ 89ᵣ│ │
│ └+────┘ └+────┘ └+────┘ └+────┘ └+────┘ └+────┘ └+─────┘ └+──────┘ └+──────┘ └+──────┘ └+──────┘ └+───────┘ │
└∊────────────────────────────────────────────────────────────────────────────────────────────────────────────┘

[12]ₓ

┌→────────────────────────────────────────────────┐
│ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ ┌→┐ │
│ │2│ │2│ │2│ │2│ │2│ │2│ │2│ │2│ │2│ │2│ │2│ │2│ │
│ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ └ₓ┘ │
└∊────────────────────────────────────────────────┘

#### 3. Mix: turn nested vectors into rows

`⊃` (**Mix**) stacks equal-length nested vectors into a matrix. First see the small example, then apply it to `fibs`.

In [23]:
%%bpl
⍝ Parentheses group strands; square brackets explicitly construct each inner vector. Both are equivalent here.
((1 2)(3 4)) ≡ [1 2][3 4]
⊃(1 2)(3 4)
⊃fibs
ρ(⊃fibs)

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

$t

┌→──┐
↓1 2│
│3 4│
└~──┘

┌→───────┐
↓  1ᵣ  0ᵣ│
│  1ᵣ  1ᵣ│
│  2ᵣ  1ᵣ│
│  3ᵣ  2ᵣ│
│  5ᵣ  3ᵣ│
│  8ᵣ  5ᵣ│
│ 13ᵣ  8ᵣ│
│ 21ᵣ 13ᵣ│
│ 34ᵣ 21ᵣ│
│ 55ᵣ 34ᵣ│
│ 89ᵣ 55ᵣ│
│144ᵣ 89ᵣ│
└+───────┘

[12 2]ₓ

#### 4. Keep the left column

`⊣/` keeps the first item of each row. First see the small example, then use it to extract the Fibonacci numbers.

In [24]:
%%bpl
⊣/[1 2◇3 4]
⊣/⊃fibs

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

1 3

1ᵣ 1ᵣ 2ᵣ 3ᵣ 5ᵣ 8ᵣ 13ᵣ 21ᵣ 34ᵣ 55ᵣ 89ᵣ 144ᵣ

#### 5. Exact big numbers, step by step

First test the pattern with $F^{10}$. We start with $F^1$, so applying `F+.×` nine more times gives $F^{10}$. Then `0 1⌷Ften` selects row 0, column 1: the 10th Fibonacci number, `55`.


In [25]:
%%bpl
Ften←F+.×⍣9 F
Ften
0 1⌷Ften

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──────┐
↓89ᵣ 55ᵣ│
│55ᵣ 34ᵣ│
└+──────┘

55ᵣ

#### 6. Compute $F_{1000}$ exactly

Now repeat the process for $F^{1000}$. Because `F` contains exact numbers (`ᵣ`), BPL retains every digit. The entry at row 0, column 1 is the 1000th Fibonacci number.

In [26]:
%%bpl
Fthousand←F+.×⍣999 F
big←0 1⌷Fthousand
big

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

43466557686937456435688527675040625802564660517371780402481729089536555417949051890403879840079255169295922593080322634775209689623239873322471161642996440906533187938298969649928516003704476137795166849228875ᵣ

#### 7. Count its digits

For a positive integer $n$, the number of decimal digits is

$$1+\lfloor\log_{10}(n)\rfloor.$$

Apply this formula to the exact value stored in `big`.

In [27]:
%%bpl
logbig←10⊛big
whole←⌊logbig
digits←1+whole
digits

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

209

- `←` stores a result under a name.
- `10⊛big` calculates $\log_{10}(big)$.
- `⌊` rounds that result down to an integer.
- Adding `1` gives the number of decimal digits.

Why? Any positive integer with $d$ digits lies between $10^{d-1}$ and $10^d$. Therefore:

$$d=1+\lfloor\log_{10}(n)\rfloor.$$

Here, `digits` is `209`, so `big` has **209 decimal digits**.

#### 8. Compare with floating point

Finally, repeat the calculation with an ordinary floating-point matrix. Compare its scientific-notation approximation with the exact integer in `big`.

In [28]:
%%bpl
Ffloat←[1 1◇1 0]
Ffloatthousand←Ffloat+.×⍣999 Ffloat
0 1⌷Ffloatthousand

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

4.346655769ₑ208

#### 9. Repeated squaring, one step at a time

First name matrix multiplication:

- `mp←+.×` means: store matrix multiplication under the name `mp`.

Now square the previous result each time:

- `F_2←F mp F` calculates $F^2$.
- `F_4←F_2 mp F_2` calculates $F^4$.
- `F_8←F_4 mp F_4` calculates $F^8$.

The exponent doubles at every squaring:

$$1\rightarrow2\rightarrow4\rightarrow8.$$

The compact expression `mp⍨⍣3 F` performs those same three squarings. Here `⍨` supplies the current matrix on both sides of `mp`.

For comparison, `F+.×⍣7 F` starts with $F^1$ and multiplies by `F` seven more times. That also reaches $F^8$, but requires seven matrix multiplications rather than three.

Each `≡` comparison should return `$t`, meaning **true**.

In [29]:
%%bpl
mp←+.×

F_2←F mp F
F_4←F_2 mp F_2
F_8←F_4 mp F_4

F_2
F_4
F_8

shortcut←mp⍨⍣3 F
F_8≡shortcut
F_8≡F+.×⍣7 F

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────┐
↓2ᵣ 1ᵣ│
│1ᵣ 1ᵣ│
└+────┘

┌→────┐
↓5ᵣ 3ᵣ│
│3ᵣ 2ᵣ│
└+────┘

┌→──────┐
↓34ᵣ 21ᵣ│
│21ᵣ 13ᵣ│
└+──────┘

$t

$t

### Markov chains: predicting the weather

A **Markov chain** is a simple way to describe something that changes at random from one day to the next. It needs two ingredients:

- **States**: the situations that can happen. Here there are two: **sunny** and **rainy**.
- **Chances**: for each state today, the chance of each state tomorrow. These chances depend only on today's weather, not on earlier days.

Our rules:

- If today is **sunny**, tomorrow is sunny 90% of the time and rainy 10% of the time.
- If today is **rainy**, tomorrow is sunny 50% of the time and rainy 50% of the time.

In the code, chances are written as decimals: 90% is `0.9`, 10% is `0.1`, and so on.

#### 1. Write the rules as a table

Put the rules in a 2×2 table called `P`. Each **row** is today's weather and each **column** is tomorrow's weather, with sunny first and rainy second:

|  | tomorrow sunny | tomorrow rainy |
|---|---|---|
| **today sunny** | 0.9 | 0.1 |
| **today rainy** | 0.5 | 0.5 |

The code does three things:

- `P←[...]` builds the table. Each line inside the brackets is one row. (`⍝` starts a comment: BPL ignores everything after it on that line.)
- `labels←[...;...]` makes the names to show. Inside these brackets, `;` separates the two lists: the row names come before it and the column names after it.
- `labels:P` uses `:` to **attach** those names to `P`, so the table prints with its rows and columns labelled. The numbers in `P` don't change; this only makes the printout easier to read.

In [30]:
%%bpl
P←[0.9 0.1    ⍝ row 0: today sunny
   0.5 0.5]   ⍝ row 1: today rainy
labels←["today sunny" "today rainy";"tomorrow sunny" "tomorrow rainy"]  ⍝ row names ; column names
labels:P      ⍝ P with labels, for display only

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────────────────────────────────────┐
↓            tomorrow sunny tomorrow rainy│
│today sunny            0.9            0.1│
│today rainy            0.5            0.5│
└~────────────────────────────────────────┘

#### 2. Check each row adds up to 1

Tomorrow must be either sunny or rainy, so the two chances in each row must add up to 1 (that is, 100%). `+/` adds across each row.

In [31]:
%%bpl
+/P

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

1 1

`1 1`: both rows add up to 1, so the table makes sense.

#### 3. Describe today as two numbers

We describe what we know about today as two chances: **chance of sun**, then **chance of rain**.

- If we know today is sunny, that is `1 0` (100% sun, 0% rain).
- If we know today is rainy, that is `0 1`.
- If it could be either, equally, that is `0.5 0.5`.

Start with a sunny day.

In [32]:
%%bpl
today←1 0   ⍝ 100% sun, 0% rain
today

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

1 0

#### 4. Work out tomorrow by hand

Because today is definitely sunny, tomorrow simply follows the **sunny row** of `P`: 0.9 sun, 0.1 rain.

When today is uncertain, mix the rows. Suppose today is `0.5 0.5`. Then tomorrow is:

- half of the sunny row: 0.5 × (0.9, 0.1) = (0.45, 0.05)
- plus half of the rainy row: 0.5 × (0.5, 0.5) = (0.25, 0.25)
- total: (0.7, 0.3)

The code does that same sum with ordinary arithmetic.

In [33]:
%%bpl
(0.5×0.9 0.1)+(0.5×0.5 0.5)   ⍝ half the sunny row + half the rainy row

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.7 0.3

#### 5. Let `+.×` do the mixing

"Multiply each row of `P` by today's chance for that row, then add the rows" is exactly what `+.×` does when a list is on the left and a table is on the right. It is the same matrix product you used in part 1.

In [34]:
%%bpl
today+.×P        ⍝ today is sunny
0.5 0.5+.×P      ⍝ today is 50/50

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.9 0.1

0.7 0.3

Both match the hand calculations: `0.9 0.1` and `0.7 0.3`.

#### 6. Give the step a name

We will move forward one day many times, so store the step as a function called `next`.

- `{...}` makes a function.
- Inside the braces, `ω` means "whatever you give the function".

So `next today` means `today+.×P`.

In [35]:
%%bpl
next←{ω+.×P}
next today   ⍝ tomorrow

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.9 0.1

#### 7. Two days ahead

To go two days ahead, apply `next` twice. BPL reads from right to left, so `next next today` means "next of (next of today)".

By hand: the day after tomorrow is sunny with chance 0.9 × 0.9 + 0.1 × 0.5 = 0.86 (sunny then sunny, or rainy then sunny).

In [36]:
%%bpl
next next today   ⍝ two days after a sunny day

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.86 0.14

#### 8. Many days at once with `⍣`

You met `⍣` (power) earlier in this part:

- `next⍣3 today` applies `next` three times: three days ahead.
- `ι6` makes the list `0 1 2 3 4 5` (you met `ι` in part 1). So `next⍣(ι6) today` keeps **every** day from day 0 (today) to day 5.
- `⊃` (Mix, from the Fibonacci steps) stacks those days into a table with one row per day.

In [37]:
%%bpl
next⍣3 today          ⍝ three days ahead
⊃next⍣(ι6) today      ⍝ days 0 to 5, one row per day

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.844 0.156

┌→──────────────┐
↓1       0      │
│0.9     0.1    │
│0.86    0.14   │
│0.844   0.156  │
│0.8376  0.1624 │
│0.83504 0.16496│
└~──────────────┘

Look down the first column (chance of sun): 1, 0.9, 0.86, 0.844, … It keeps falling, but by less each day.

#### 9. Where does it settle?

If you keep going, the numbers stop changing. `≡` (Match) asks "are these two things the same?". So `⍣≡` means "keep applying `next` until the answer is the same as last time". The answer it settles on is called the **steady state**.

In [38]:
%%bpl
s←next⍣≡ today   ⍝ keep going until nothing changes
s
next s           ⍝ one more day: still the same

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.8333333333 0.1666666667

0.8333333333 0.1666666667

In the long run, about 83% of days are sunny and 17% are rainy. Applying `next` to `s` gives `s` back, which is exactly what "steady" means.

#### 10. Does the starting day matter?

Try starting from a rainy day, `0 1`, instead.

In [39]:
%%bpl
next⍣≡ 0 1   ⍝ start from a rainy day

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0.8333333333 0.1666666667

It's the same answer. Wherever you start, this chain settles on the same steady state.

#### 11. The exact answer: set up the equations

We can also find the steady state exactly. Call the steady chance of sun `a` and the steady chance of rain `b`.

**Fact 1 – it is steady.** Tomorrow's chance of sun equals today's:

a = 0.9a + 0.5b

Take a away from both sides and tidy up:

0.1a − 0.5b = 0

**Fact 2 – the chances add up to 1:**

a + b = 1

That is two equations with two unknowns, the same kind of problem `⌹` solved in part 1.

#### 12. Solve the equations with `⌹`

Write the equations as a table `E`, with one row per equation: the number in front of `a`, then the number in front of `b`. The right-hand sides, 0 and 1, go on the left of `⌹`. As in part 1, `b⌹E` finds the unknowns that make `E` times those unknowns equal `b`.

- `1r10` is the exact fraction 1/10.
- `¯1r2` is −1/2 (BPL writes negative numbers with `¯`).
- `ᵣ` keeps every number exact.

In [40]:
%%bpl
E←[1r10 ¯1r2   ⍝ 0.1a − 0.5b = 0
   1    1]ᵣ    ⍝ a + b = 1
E
[0 1]ᵣ⌹E       ⍝ solve for a and b

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────┐
↓1ᵣ10 ¯1ᵣ2│
│  1ᵣ   1ᵣ│
└+────────┘

5ᵣ6 1ᵣ6

The answer is exactly 5/6 (about 0.833) and 1/6 (about 0.167), which matches step 9.